# K-means

In [12]:
# python 
import os
os.environ["OMP_NUM_THREADS"] = "3"
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_samples, silhouette_score
import matplotlib.pyplot as plt



df = pd.read_csv("./datasets/comportamiento_jugadores.csv")
df.head(100)


,Metrica_X,Metrica_Y,Escenario,Descripcion_Ejes
0,44.125085,74.436148,1_Economia_Juego,X: Horas_Semana | Y: Gasto_USD
1,19.905609,115.313604,1_Economia_Juego,X: Horas_Semana | Y: Gasto_USD
2,55.833550,119.616643,1_Economia_Juego,X: Horas_Semana | Y: Gasto_USD
3,10.243052,129.259511,1_Economia_Juego,X: Horas_Semana | Y: Gasto_USD
4,5.413660,259.896958,1_Economia_Juego,X: Horas_Semana | Y: Gasto_USD
...,...,...,...,...
95,49.823761,136.865158,1_Economia_Juego,X: Horas_Semana | Y: Gasto_USD
96,35.114618,121.159379,1_Economia_Juego,X: Horas_Semana | Y: Gasto_USD
97,22.449715,102.720996,1_Economia_Juego,X: Horas_Semana | Y: Gasto_USD
98,22.435696,114.961632,1_Economia_Juego,X: Horas_Semana | Y: Gasto_USD


In [13]:
# Filtrado
df_1_Economia_Juego = df[df["Escenario"] == "1_Economia_Juego"]
df_2_Deteccion_Hackers = df[df["Escenario"] == "2_Deteccion_Hackers"]
df_3_Estilos_Batalla = df[df["Escenario"] == "3_Estilos_Batalla"]

# Verification
# print("------------\n",df_1_Economia_Juego.head(5))
# print("------------\n",df_2_Deteccion_Hackers.head(5))
# print("------------\n",df_3_Estilos_Batalla.head(5))

# Extraer métricas
df_1_Economia_Juego_raw = df_1_Economia_Juego[["Metrica_X","Metrica_Y"]]
df_2_Deteccion_Hackers_raw = df_2_Deteccion_Hackers[["Metrica_X","Metrica_Y"]]
df_3_Estilos_Batalla_raw = df_3_Estilos_Batalla[["Metrica_X","Metrica_Y"]]

scaler_1, scaler_2, scaler_3 = StandardScaler(), StandardScaler(), StandardScaler()

# Escalamiento
df_1_scaled = scaler_1.fit_transform(df_1_Economia_Juego_raw)
df_2_scaled = scaler_2.fit_transform(df_2_Deteccion_Hackers_raw)
df_3_scaled = scaler_3.fit_transform(df_3_Estilos_Batalla_raw)

In [14]:
# Selección del numero de clusters

# Posible rango de clusters
range_n_clusters = range(2, 9)
datasets = {
    "df_Economia_Juego_Scaled": df_1_scaled, 
    "df_Deteccion_Hackers_Scaled": df_2_scaled,
    "df_Estilos_Batalla_Scaled": df_3_scaled
}

''' 
Aqui a lo mejor podrían hacer una grafica como las que tiene aca
https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html#sphx-glr-auto-examples-cluster-plot-kmeans-silhouette-analysis-py
'''

resultados = []
for nombre, datos in datasets.items():
    for i in range_n_clusters:
        # Inicilialización del clusterer con n_clusters
        clusterer = KMeans(n_clusters=i, random_state=10, n_init=10)
        cluster_labels = clusterer.fit_predict(datos)

        # El silhouette_score nos da el promedio para todas las muestras
        # Nos ayuda a visualizar la densidad y separación de los clusters creados
        silhouette_avg = silhouette_score(datos, cluster_labels)

        resultados.append({
            "dataset": nombre,
            "n_clusters": i,
            "inercia": clusterer.inertia_,
            "silhouette": silhouette_avg
        })

df_resultados = pd.DataFrame(resultados)
print(df_resultados)
print(
    df_resultados.sort_values("silhouette", ascending=False)
    .groupby("dataset").head(1).sort_values("dataset")
)

                        dataset  n_clusters     inercia  silhouette
0      df_Economia_Juego_Scaled           2  309.952133    0.678296
1      df_Economia_Juego_Scaled           3   71.873618    0.737604
2      df_Economia_Juego_Scaled           4   62.534017    0.604248
3      df_Economia_Juego_Scaled           5   53.877370    0.484153
4      df_Economia_Juego_Scaled           6   46.892207    0.322082
5      df_Economia_Juego_Scaled           7   40.581802    0.340328
6      df_Economia_Juego_Scaled           8   35.126389    0.347349
7   df_Deteccion_Hackers_Scaled           2  502.641571    0.494194
8   df_Deteccion_Hackers_Scaled           3  369.783088    0.442570
9   df_Deteccion_Hackers_Scaled           4  245.653731    0.437400
10  df_Deteccion_Hackers_Scaled           5  181.050728    0.463206
11  df_Deteccion_Hackers_Scaled           6  138.036082    0.476758
12  df_Deteccion_Hackers_Scaled           7  109.674004    0.495977
13  df_Deteccion_Hackers_Scaled           8   84

In [15]:
# Modelos
km_economia_juego = KMeans(n_clusters=7, random_state=10, n_init=10).fit_predict(df_1_scaled)
km_deteccion_hackers = KMeans(n_clusters=3, random_state=10, n_init=10).fit_predict(df_2_scaled)
km_estilos_batalla = KMeans(n_clusters=7, random_state=10, n_init=10).fit_predict(df_3_scaled)